## Classification Models

## Logistic Regression

- Sigmoid function: converts the linear output into a probability
- Log loss: the right cost function for classification

In [35]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [36]:
print(sigmoid(-10))   # very negative input: expect a value very close to 0
print(sigmoid(0))     # expect: exactly 0.5
print(sigmoid(10))    # very positive input: expect a value very close to 1

4.5397868702434395e-05
0.5
0.9999546021312976


In [37]:
import numpy as np

class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.lr = learning_rate
        self.epochs = epochs
        self.w = None
        self.b = None
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0
        
        for epoch in range(self.epochs):
            # Step 1: calculate the linear part
            z = X @ self.w + self.b  # hint: X @ self.w + self.b
            
            # Step 2: apply the sigmoid to get probabilities
            y_pred = sigmoid(z)  # hint: sigmoid(z)
            
            # Step 3: calculate the gradients (formula above)
            dw = (1/n_samples) * X.T @ (y_pred - y)
            db = (1/n_samples) * np.sum(y_pred - y)
            
            # Step 4: update the weights
            self.w -= self.lr * dw
            self.b -= self.lr * db
    
    def predict_proba(self, X):
        # Returns the probability (between 0 and 1)
        z = X @ self.w + self.b
        return sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        # Returns the class label (0 or 1)
        probabilities = self.predict_proba(X)
        return (probabilities >= threshold).astype(int)

In [38]:
# Simple dataset: "time_spent_on_site" (minutes) → "purchased" (0/1)
X = np.array([[1], [2], [3], [4], [5], [6], [7], [8], [9], [10]])
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])  # clear pattern: <5.5 min = No, >5.5 min = Yes

model = LogisticRegressionScratch(learning_rate=0.1, epochs=5000)
model.fit(X, y)

print("Predicted probabilities:", model.predict_proba(X).round(3))
print("Predicted classes:", model.predict(X))
print("Actual classes:", y)

Predicted probabilities: [0.    0.001 0.008 0.056 0.303 0.761 0.959 0.994 0.999 1.   ]
Predicted classes: [0 0 0 0 0 1 1 1 1 1]
Actual classes: [0 0 0 0 0 1 1 1 1 1]


In [39]:
from sklearn.linear_model import LogisticRegression

sklearn_model = LogisticRegression()
sklearn_model.fit(X, y)

probabilities = sklearn_model.predict_proba(X)[:, 1]  # probability of class "1"
predictions = sklearn_model.predict(X)

print("sklearn probabilities:", probabilities.round(3))
print("sklearn predictions:", predictions)

sklearn probabilities: [0.005 0.016 0.049 0.144 0.356 0.644 0.856 0.951 0.985 0.995]
sklearn predictions: [0 0 0 0 0 1 1 1 1 1]


In [40]:
from sklearn.linear_model import LogisticRegression

sklearn_model = LogisticRegression()
sklearn_model.fit(X, y)

probabilities = sklearn_model.predict_proba(X)[:, 1]  # probability of class "1"
predictions = sklearn_model.predict(X)

print("sklearn probabilities:", probabilities.round(3))
print("sklearn predictions:", predictions)

sklearn probabilities: [0.005 0.016 0.049 0.144 0.356 0.644 0.856 0.951 0.985 0.995]
sklearn predictions: [0 0 0 0 0 1 1 1 1 1]


## K-Nearest Neighbour KNN

In [41]:
import numpy as np
from collections import Counter

class KNNClassifierScratch:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        # Remember: in KNN, "fit" only STORES the data; nothing is trained
        self.X_train = X
        self.y_train = y
    
    def euclidean_distance(self, point1, point2):
        return np.sqrt(np.sum((point1 - point2) ** 2))
    
    def predict_single(self, x):
        # Step 1: calculate the distance from the new point 'x' to all training points
        distances = [self.euclidean_distance(x, x_train) for x_train in self.X_train]  
        
        # Step 2: find the indices of the k smallest distances
        k_indices = np.argsort(distances)[:self.k] 
        
        # Step 3: get the labels for these indices
        k_nearest_labels = [self.y_train[i] for i in k_indices] 

        # Step 4: find the majority class (the label that appears most often)
        most_common = Counter(k_nearest_labels).most_common(1)[0][0] 
        
        return most_common
    
    def predict(self, X):
        return np.array([self.predict_single(x) for x in X])

In [42]:
import numpy as np
import pandas as pd

# Set seed for reproducibility
np.random.seed(42)
n_samples = 100

# 1. Feature Generation
time_spent_on_site = np.random.normal(loc=12, scale=5, size=n_samples)  # in minutes
time_spent_on_site = np.clip(time_spent_on_site, 1, 45).round(2)

pages_visited = np.random.poisson(lam=4, size=n_samples) + 1  # minimum 1 page

previous_purchases = np.random.choice(
    [0, 1, 2, 3, 4, 5], 
    size=n_samples, 
    p=[0.45, 0.25, 0.15, 0.08, 0.04, 0.03]
)

# 2. Linear Combination (Log-Odds)
# B0 (intercept) + B1*time + B2*pages + B3*purchases
log_odds = (
    -3.5 
    + 0.12 * time_spent_on_site 
    + 0.35 * pages_visited 
    + 0.60 * previous_purchases
)

# 3. Probability via Sigmoid Function: 1 / (1 + e^(-log_odds))
probabilities = 1 / (1 + np.exp(-log_odds))

# 4. Generate Binary Target Variable
will_purchase = np.random.binomial(1, probabilities)

# 5. Assemble DataFrame
df = pd.DataFrame({
    'time_spent_on_site': time_spent_on_site,
    'pages_visited': pages_visited,
    'previous_purchases': previous_purchases,
    'will_purchase': will_purchase
})

# Display summary
print(df.head())
print("\nTarget Class Distribution:")
print(df['will_purchase'].value_counts(normalize=True))

   time_spent_on_site  pages_visited  previous_purchases  will_purchase
0               14.48              3                   0              0
1               11.31              6                   0              0
2               15.24              6                   2              1
3               19.62              3                   3              1
4               10.83              2                   1              0

Target Class Distribution:
will_purchase
0    0.57
1    0.43
Name: proportion, dtype: float64


In [43]:
X = df.drop(columns=['will_purchase']).values
y = df['will_purchase'].values

In [44]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [45]:
for k in [1, 3, 5, 10, 20, 50]:
    knn = KNNClassifierScratch(k=k)
    knn.fit(X_train_scaled, y_train)
    preds = knn.predict(X_test_scaled)
    accuracy = np.mean(preds == y_test)
    print(f"k={k}: Accuracy = {accuracy:.2%}")

k=1: Accuracy = 50.00%
k=3: Accuracy = 65.00%
k=5: Accuracy = 75.00%
k=10: Accuracy = 70.00%
k=20: Accuracy = 80.00%
k=50: Accuracy = 65.00%


In [46]:
from sklearn.neighbors import KNeighborsClassifier

for k in [1, 3, 5, 10, 20, 50]:
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(X_train_scaled, y_train)

    predictions = knn_model.predict(X_test_scaled)
    probabilities = knn_model.predict_proba(X_test_scaled)  # KNN also supports predict_proba

    accuracy = knn_model.score(X_test_scaled, y_test)
    print(f"k ={k}: Accuracy = {accuracy:.2%}")

k =1: Accuracy = 50.00%
k =3: Accuracy = 65.00%
k =5: Accuracy = 75.00%
k =10: Accuracy = 65.00%
k =20: Accuracy = 75.00%
k =50: Accuracy = 60.00%


## Navie Bayes

In [47]:
import numpy as np

class GaussianNaiveBayesScratch:
    def __init__(self):
        self.class_priors = {}      # P(Class)
        self.means = {}              # mean of each feature for each class
        self.variances = {}          # variance of each feature for each class
        self.classes = None
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        n_samples = X.shape[0]
        
        for c in self.classes:
            X_c = X[y == c]   # samples of class 'c' only
            
            # Step 1: calculate the prior probability P(Class=c)
            self.class_priors[c] = len(X_c)/n_samples
            
            # Step 2: calculate the mean and variance of each feature for this class
            self.means[c] = X_c.mean(axis=0)      
            self.variances[c] = X_c.var(axis =0)

    def gaussian_probability(self, x, mean, var):
        # Calculate the probability density for each feature
        eps = 1e-9  # small smoothing value to avoid division by zero
        numerator = np.exp(-((x - mean) ** 2) / (2 * var + eps))
        denominator = np.sqrt(2 * np.pi * var + eps)
        return numerator / denominator
    
    def predict_single(self, x):
        class_scores = {}
        
        for c in self.classes:
            # Step 1: start with the prior probability (in log space, so multiplication becomes addition)
            score = np.log(self.class_priors[c])
            
            # Step 2: calculate each feature's probability density and add its log (log space: multiplication = addition)
            probabilities = self.gaussian_probability(x, self.means[c], self.variances[c])
            score += np.sum(np.log(probabilities) + 1e-9)  # hint: np.sum(np.log(probabilities + 1e-9))
            
            class_scores[c] = score
        
        # Step 3: return the class with the highest score
        return max(class_scores, key=class_scores.get)  # hint: max(class_scores, key=class_scores.get)
    
    def predict(self, X):
        return np.array([self.predict_single(x) for x in X])

In [48]:
import numpy as np

np.random.seed(15)
n_samples = 300

annual_income = np.random.normal(50000, 20000, n_samples).clip(10000, 150000)
credit_score = np.random.normal(650, 100, n_samples).clip(300, 850)
loan_amount = np.random.normal(20000, 10000, n_samples).clip(1000, 60000)
existing_debts = np.random.poisson(2, n_samples)

# True relationship: high income + high credit score + low loan/debt = low default risk
risk_score = (
    -0.00003 * annual_income 
    - 0.008 * credit_score 
    + 0.00005 * loan_amount 
    + 0.3 * existing_debts
    + 5
)
default_prob = 1 / (1 + np.exp(-risk_score))
will_default = np.random.binomial(1, default_prob)

X = np.column_stack([annual_income, credit_score, loan_amount, existing_debts])
y = will_default

print(f"Default rate: {y.mean():.1%}")

Default rate: 51.0%


In [49]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

In [50]:
# from scratch implementation

model = GaussianNaiveBayesScratch()

model.fit(X_train, y_train)

pred = model.predict(X_test)

accuracy = np.mean(pred == y_test)

print('Predictions:', pred)
print('Actual:', y_test)
print(f'Accuracy: {accuracy:.2%}')

Predictions: [1 1 1 0 0 0 1 0 1 0 0 0 1 0 0 0 1 0 1 1 1 0 1 0 1 0 1 0 0 0 1 0 1 1 0 1 1
 1 1 1 0 0 0 1 0 1 1 1 0 0 1 1 0 0 0 1 1 0 0 0]
Actual: [1 1 1 1 1 0 1 0 1 0 0 0 1 0 0 1 0 1 1 1 1 1 0 1 1 0 1 0 0 1 1 0 0 1 1 0 1
 1 1 1 1 0 0 1 0 1 1 1 1 1 1 1 0 0 0 1 1 0 0 0]
Accuracy: 75.00%


In [51]:
# sklearn implementation 
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)
nb_prob = nb.predict_proba(X_test)

nb_accuracy = nb.score(X_test, y_test)

print(f"sklearn Accuracy: {accuracy:.2%}")

sklearn Accuracy: 75.00%


## SVM

In [1]:
import numpy as np

class SVMScratch:
    def __init__(self, learning_rate=0.001, epochs=1000, lambda_param=0.01):
        self.lr = learning_rate
        self.epochs = epochs
        self.lambda_param = lambda_param  # regularization strength (margin width control)
        self.w = None
        self.b = None
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        y_ = np.where(y <= 0, -1, 1)  # convert labels to -1/+1
        
        self.w = np.zeros(n_features)
        self.b = 0
        
        for epoch in range(self.epochs):
            for idx, x_i in enumerate(X):
                condition = y_[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                
                if condition:
                    # Point is on the correct side, outside the margin: only the regularisation gradient applies
                    dw = 2 * self.lambda_param * self.w
                    db = 0
                else:
                    # Point is inside the margin or misclassified: the hinge-loss gradient is added too
                    dw = 2 * self.lambda_param * self.w - np.dot(x_i, y_[idx])
                    db = y_[idx]
                
                self.w -= self.lr * dw
                self.b -= self.lr * db
    
    def predict(self, X):
        approx = np.dot(X, self.w) - self.b
        return np.where(approx >= 0, 1, 0)  # convert back to 0/1

In [2]:
import numpy as np
import pandas as pd

# Set seed for reproducibility
np.random.seed(42)
n_samples = 150

# 1. Feature Generation
time_spent_on_site = np.random.normal(loc=12, scale=5, size=n_samples)  # in minutes
time_spent_on_site = np.clip(time_spent_on_site, 1, 45).round(2)

pages_visited = np.random.poisson(lam=4, size=n_samples) + 1  # minimum 1 page

previous_purchases = np.random.choice(
    [0, 1, 2, 3, 4, 5], 
    size=n_samples, 
    p=[0.45, 0.25, 0.15, 0.08, 0.04, 0.03]
)

# 2. Linear Combination (Log-Odds)
# B0 (intercept) + B1*time + B2*pages + B3*purchases
log_odds = (
    -3.5 
    + 0.12 * time_spent_on_site 
    + 0.35 * pages_visited 
    + 0.60 * previous_purchases
)

# 3. Probability via Sigmoid Function: 1 / (1 + e^(-log_odds))
probabilities = 1 / (1 + np.exp(-log_odds))

# 4. Generate Binary Target Variable
will_purchase = np.random.binomial(1, probabilities)

# 5. Assemble DataFrame
df = pd.DataFrame({
    'time_spent_on_site': time_spent_on_site,
    'pages_visited': pages_visited,
    'previous_purchases': previous_purchases,
    'will_purchase': will_purchase
})

X = df.drop(columns=['will_purchase']).values
y = df['will_purchase'].values

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVMScratch(learning_rate=0.001, epochs=1000, lambda_param=0.01)
svm.fit(X_train_scaled, y_train)

predictions = svm.predict(X_test_scaled)
accuracy = np.mean(predictions == y_test)
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 70.00%


In [4]:
from sklearn.svm import SVC

svm_rbf = SVC(kernel='rbf', gamma='scale', C=1.0)
svm_rbf.fit(X_train_scaled, y_train)

pred = svm_rbf.predict(X_test_scaled)

model_accuracy = svm_rbf.score(X_test_scaled, y_test)

print(f"Accuracy: {model_accuracy:.2%}")

Accuracy: 83.33%


In [5]:
from sklearn.svm import SVC
import numpy as np

results = []
for C in [0.01, 0.1, 1, 10, 100]:
    for gamma in [0.001, 0.01, 0.1, 1]:
        model = SVC(kernel='rbf', C=C, gamma=gamma, random_state=42)
        model.fit(X_train_scaled, y_train)
        
        train_acc = model.score(X_train_scaled, y_train)
        test_acc = model.score(X_test_scaled, y_test)
        
        results.append({'C': C, 'gamma': gamma, 'train_acc': train_acc, 'test_acc': test_acc})

import pandas as pd
df = pd.DataFrame(results)
top5 = df.sort_values('test_acc', ascending=False).head(5)
print(top5)

        C  gamma  train_acc  test_acc
14   10.0  0.100   0.716667  0.800000
10    1.0  0.100   0.625000  0.800000
16  100.0  0.001   0.583333  0.766667
17  100.0  0.010   0.683333  0.733333
13   10.0  0.010   0.583333  0.733333


In [6]:
from sklearn.svm import SVC
import numpy as np

results = []
for C in [0.01, 0.1, 1, 10, 100]:
    for gamma in [0.001, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 1, 'scale']:
        model = SVC(kernel='rbf', C=C, gamma=gamma, random_state=42)
        model.fit(X_train_scaled, y_train)
        
        train_acc = model.score(X_train_scaled, y_train)
        test_acc = model.score(X_test_scaled, y_test)
        
        results.append({'C': C, 'gamma': gamma, 'train_acc': train_acc, 'test_acc': test_acc})

import pandas as pd
df = pd.DataFrame(results)
top5 = df.sort_values('test_acc', ascending=False).head(5)
print(top5)

        C  gamma  train_acc  test_acc
26    1.0  scale   0.725000  0.833333
29   10.0   0.05   0.691667  0.833333
38  100.0   0.05   0.725000  0.833333
23    1.0    0.3   0.725000  0.833333
21    1.0    0.1   0.625000  0.800000


In [7]:
# Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier

tree_clf = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
tree_clf.fit(X_train_scaled, y_train)  # scaling is not required for tree models, but kept for consistency

predictions = tree_clf.predict(X_test_scaled)
probabilities = tree_clf.predict_proba(X_test_scaled)  # class probabilities are also available
accuracy = tree_clf.score(X_test_scaled, y_test)

In [8]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=150, max_depth=6, max_features='sqrt', random_state=42)
rf_clf.fit(X_train, y_train)

,n_estimators,150
,criterion,'gini'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [9]:
# Gradient Boosting Classifier

from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_clf.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [20]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42),
    'SVM': SVC(kernel='rbf', gamma='scale', C=1.0, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Gaussian Naive Bayes': GaussianNB()
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    accuracy = model.score(X_test_scaled, y_test)
    print(f"{name}: {accuracy:.2%}")

Decision Tree: 66.67%
Random Forest: 70.00%
Gradient Boosting: 80.00%
Logistic Regression: 73.33%
SVM: 83.33%
K-Nearest Neighbors: 70.00%
Gaussian Naive Bayes: 76.67%
